# Content-Based Anime Recommender

**Goal:** Build a system that recommends anime similar to what you like

## How it works:
1. Load our engineered features
2. Calculate similarity between all anime
3. Given an anime, find the most similar ones
4. Return top N recommendations

---

In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

Libraries imported!


In [20]:
X_features = pd.read_csv('../data/processed/X_features_only.csv')
titles = pd.read_csv('../data/processed/anime_titles.csv')

print(f"Loaded data!")
print(f"Shape: {X_features.shape[0]} anime × {X_features.shape[1]} features")
print(f"\nFirst 5 anime:")
print(titles.head())

Loaded data!
Shape: 448 anime × 107 features

First 5 anime:
   index                             title
0      0                Shingeki no Kyojin
1      1                        Death Note
2      2  Fullmetal Alchemist: Brotherhood
3      3                     One Punch Man
4      4                  Kimetsu no Yaiba


## Understanding Our Features

Our features look like this:

| Title | score_norm | Action | Comedy | Drama | ... |
|-------|-----------|--------|--------|-------|-----|
| Naruto | 0.85 | 1 | 0 | 1 | ... |
| Death Note | 0.90 | 0 | 0 | 1 | ... |

Each anime is represented as a **vector** (list of numbers).

When two anime have similar numbers, they're similar!

In [21]:
print("Calculating similarity between all anime...")
print("(This might take 10-20 seconds)")

similarity_matrix = cosine_similarity(X_features)

print(f"\n✅ Done!")
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"   → {similarity_matrix.shape[0]} anime × {similarity_matrix.shape[1]} anime")

Calculating similarity between all anime...
(This might take 10-20 seconds)

✅ Done!
Similarity matrix shape: (448, 448)
   → 448 anime × 448 anime


In [22]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=titles['title'],
    columns=titles['title']
)

print("Example: How similar is the first anime to others?")
print(f"\nAnime: {titles['title'].iloc[0]}")
print("\nTop 5 most similar anime:")

# Get first row, sorted by similarity, show top 5
first_anime_similarities = similarity_df.iloc[0].sort_values(ascending=False).head(6)
print(first_anime_similarities)

Example: How similar is the first anime to others?

Anime: Shingeki no Kyojin

Top 5 most similar anime:
title
Shingeki no Kyojin                             1.000000
Shingeki no Kyojin Season 3 Part 2             0.815907
Shingeki no Kyojin Season 3                    0.790664
Shingeki no Kyojin Season 2                    0.789583
Shingeki no Kyojin: The Final Season           0.742701
Shingeki no Kyojin: The Final Season Part 2    0.714193
Name: Shingeki no Kyojin, dtype: float64


In [23]:
def get_recommendations(anime_title, n_recommendations=10):
    """ Get anime recommendations based on similarity """
    
    try:
        anime_idx = titles[titles['title'] == anime_title].index[0]
    except IndexError:
        print(f"❌ Error: '{anime_title}' not found in database!")
        print(f"\nTip: Try searching for it first using search_anime()")
        return None
    
    similarity_scores = similarity_df.iloc[anime_idx]
    similarity_scores = similarity_scores.sort_values(ascending=False)
    
    # Get top N (excluding the anime itself, which is always #1)
    top_similar = similarity_scores.iloc[1:n_recommendations+1]
    
    recommendations = pd.DataFrame({
        'Rank': range(1, len(top_similar) + 1),
        'Anime': top_similar.index,
        'Similarity Score': top_similar.values,
        'Match %': (top_similar.values * 100).round(1)
    })
    
    return recommendations

print("Recommender function created!")

Recommender function created!


In [24]:
def search_anime(keyword):
    """ Search for anime titles containing a keyword """
    keyword = keyword.lower()
    matches = titles[titles['title'].str.lower().str.contains(keyword)]
    
    if len(matches) == 0:
        print(f"❌ No anime found containing '{keyword}'")
        return None
    
    print(f"Found {len(matches)} anime matching '{keyword}':")
    for idx, row in matches.iterrows():
        print(f"  -> {row['title']}")
    
    return matches['title'].tolist()

print("Search function created!")

Search function created!


In [25]:
print("=" * 70)
print("ANIME RECOMMENDER SYSTEM")
print("=" * 70)

test_anime = titles['title'].iloc[0] 

print(f"\nYou liked: {test_anime}")
print("\nRecommendations:\n")

recommendations = get_recommendations(test_anime, n_recommendations=10)
print(recommendations.to_string(index=False))

print("\n" + "=" * 70)

ANIME RECOMMENDER SYSTEM

You liked: Shingeki no Kyojin

Recommendations:

 Rank                                       Anime  Similarity Score  Match %
    1          Shingeki no Kyojin Season 3 Part 2          0.815907     81.6
    2                 Shingeki no Kyojin Season 3          0.790664     79.1
    3                 Shingeki no Kyojin Season 2          0.789583     79.0
    4        Shingeki no Kyojin: The Final Season          0.742701     74.3
    5 Shingeki no Kyojin: The Final Season Part 2          0.714193     71.4
    6                            Kimetsu no Yaiba          0.627269     62.7
    7                                Vinland Saga          0.601074     60.1
    8                              Akame ga Kill!          0.575919     57.6
    9            Fullmetal Alchemist: Brotherhood          0.565389     56.5
   10                            Mirai Nikki (TV)          0.529797     53.0



In [26]:
def explain_recommendation(source_anime, recommended_anime):
    """ Explain why an anime was recommended """
    source_idx = titles[titles['title'] == source_anime].index[0]
    rec_idx = titles[titles['title'] == recommended_anime].index[0]
    
    source_features = X_features.iloc[source_idx]
    rec_features = X_features.iloc[rec_idx]
    
    shared_features = []
    for col in X_features.columns:
        if source_features[col] == 1 and rec_features[col] == 1:
            shared_features.append(col)
    
    similarity = similarity_df.loc[source_anime, recommended_anime]
    
    print(f"   Why we recommended '{recommended_anime}'")
    print(f"   based on '{source_anime}':\n")
    print(f"   Similarity Score: {similarity:.2%}\n")
    print(f"   Shared features ({len(shared_features)}):")
    
    for feature in shared_features[:10]:
        print(f"      -> {feature}")
    
    if len(shared_features) > 10:
        print(f"      ... and {len(shared_features) - 10} more")

print("=" * 70)
test_source = titles['title'].iloc[0]
recs = get_recommendations(test_source, n_recommendations=3)
if recs is not None and len(recs) > 0:
    test_rec = recs['Anime'].iloc[0]
    explain_recommendation(test_source, test_rec)
print("=" * 70)

   Why we recommended 'Shingeki no Kyojin Season 3 Part 2'
   based on 'Shingeki no Kyojin':

   Similarity Score: 81.59%

   Shared features (10):
      -> Action
      -> Drama
      -> Gore
      -> Military
      -> Shounen
      -> Survival
      -> Suspense
      -> score_cat_Excellent
      -> era_Modern
      -> has_war


In [27]:
import pickle

recommender_system = {
    'similarity_matrix': similarity_matrix,
    'similarity_df': similarity_df,
    'titles': titles,
    'X_features': X_features,
}

output_path = Path('../models')
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / 'content_recommender.pkl', 'wb') as f:
    pickle.dump(recommender_system, f)

print("Recommender system saved to: ../models/content_recommender.pkl")
print(f"Contains:")
print(f"   - Similarity matrix {similarity_matrix.shape}")
print(f"   - {len(titles)} anime titles")
print(f"   - {X_features.shape[1]} features per anime")

Recommender system saved to: ../models/content_recommender.pkl
Contains:
   - Similarity matrix (448, 448)
   - 448 anime titles
   - 107 features per anime


## Content-Based Recommender Complete!

### What I built:
1. Loaded engineered features
2. Calculated similarity between all anime (448×448 matrix)
3. Created recommendation function
4. Added search functionality
5. Built explanation system
6. Saved model for deployment

### How to use:
```python
# Search for an anime
search_anime("naruto")

# Get recommendations
recommendations = get_recommendations("Naruto", n_recommendations=10)
```

### Performance:
- **Speed:** Instant recommendations (pre-calculated similarity)
- **Accuracy:** Based on 100+ features per anime
- **Coverage:** Can recommend from all anime